# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "restaurant.db"

OpenAI API Key exists and begins sk-proj-


In [ ]:
# with sqlite3.connect(DB) as conn:
#     cursor = conn.cursor()
#     cursor.execute('CREATE TABLE IF NOT EXISTS menu (name TEXT PRIMARY KEY, price REAL)')
#     conn.commit()

In [3]:
## Restorant Assistant

restaurant_system_message = """
You are a restaurant assistant for help customer to answer question and upselling product.
Be kind and friendly.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def get_all_menu():
    print(f"DATABASE TOOL CALED: Getting all menu and price")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * from menu')
        result = cursor.fetchall()
        menu = [
            {
                "name": res[0],
                "price": res[1]
            }
            for res in result
        ]
        # return f"This is all menu in our DB: {menu}"
        return json.dumps(menu, ensure_ascii=False)

In [5]:
def add_menu(name, price):
    print(f"DATABASE TOOL CALED: Insert new menu in DB")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO menu (name, price) VALUES (?, ?) ON CONFLICT(name) DO UPDATE SET price = ?', (name.lower(), price, price))
        conn.commit()
        return f"New menu {name} was submited with {price} price."

In [ ]:
add_menu("es teh", 3000)

In [6]:
get_all_menu()

DATABASE TOOL CALED: Getting all menu and price


'[{"name": "ayam geprek", "price": 10000.0}, {"name": "es teh", "price": 3000.0}, {"name": "ayam geprek krisbar", "price": 12000.0}, {"name": "es kelapa muda", "price": 8000.0}, {"name": "jus melon", "price": 6000.0}]'

In [7]:
def get_price_by_name(name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price from menu where name = ?", (name.lower(),))
        result = cursor.fetchone()
        return f'Prices of {name} is Rp {result[0]}' if result else "No price data avaliable for this menu name"

In [8]:
get_price_by_name("ayam geprek")

'Prices of ayam geprek is Rp 10000.0'

In [9]:
get_all_menu_function = {
    "name": "get_all_menu",
    "description": "Get all data menu of a return name and price",
    "parameters": {}
}

tools = [{"type": "function", "function": get_all_menu_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_all_menu',
   'description': 'Get all data menu of a return name and price',
   'parameters': {}}}]

In [10]:
get_price_by_name_function = {
    "name": "get_price_by_name",
    "description": "Get the price of a return menu name",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The name that the customer wants to order",
            }
        },
        "required": ["name"],
        "additionalProperties": False
    }
}

tools.append({"type": "function", "function": get_price_by_name_function})
tools

[{'type': 'function',
  'function': {'name': 'get_all_menu',
   'description': 'Get all data menu of a return name and price',
   'parameters': {}}},
 {'type': 'function',
  'function': {'name': 'get_price_by_name',
   'description': 'Get the price of a return menu name',
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'The name that the customer wants to order'}},
    'required': ['name'],
    'additionalProperties': False}}}]

In [11]:
add_menu_function = {
    "name": "add_menu",
    "description": "Insert new menu name and price to DB",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The new name of menu that owner wants to add"
            },
            "price": {
                "type": "integer",
                "description": "The prices of the new menu name"
            }
        },
        "required": ["name", "price"],
        "additionalProperties": False
    }
}

tools.append({"type": "function", "function": add_menu_function})

In [12]:
tools

[{'type': 'function',
  'function': {'name': 'get_all_menu',
   'description': 'Get all data menu of a return name and price',
   'parameters': {}}},
 {'type': 'function',
  'function': {'name': 'get_price_by_name',
   'description': 'Get the price of a return menu name',
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'The name that the customer wants to order'}},
    'required': ['name'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'add_menu',
   'description': 'Insert new menu name and price to DB',
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'The new name of menu that owner wants to add'},
     'price': {'type': 'integer',
      'description': 'The prices of the new menu name'}},
    'required': ['name', 'price'],
    'additionalProperties': False}}}]

In [38]:
def chat(message, history):
    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = [
        {"role": "system", "content": restaurant_system_message}
    ] + history + [
        {"role": "user", "content": message}
    ]

    while True:

        print("CALLING OPENAI...")

        stream = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            stream=True
        )

        print(stream)

        response = ""
        tool_calls = {}

        for chunk in stream:

            print("CHUNK:", chunk)

            delta = chunk.choices[0].delta

            # =========================
            # TEXT
            # =========================
            if delta.content:
                response += delta.content
                print("TEXT:", response)

                yield response

            # =========================
            # TOOL CALL
            # =========================
            if delta.tool_calls:

                print("TOOL CALL:", delta.tool_calls)

                for tool_call in delta.tool_calls:

                    index = tool_call.index

                    if index not in tool_calls:
                        tool_calls[index] = {
                            "id": "",
                            "name": "",
                            "arguments": ""
                        }

                    if tool_call.id:
                        tool_calls[index]["id"] += tool_call.id

                    if tool_call.function:

                        if tool_call.function.name:
                            tool_calls[index]["name"] += (
                                tool_call.function.name
                            )

                        if tool_call.function.arguments:
                            tool_calls[index]["arguments"] += (
                                tool_call.function.arguments
                            )

        print("STREAM FINISHED")
        print("TOOL CALLS:", tool_calls)

        # =========================
        # TIDAK ADA TOOL
        # =========================
        if not tool_calls:
            print("NO TOOL CALL -> DONE")
            break

        # =========================
        # ASSISTANT MESSAGE
        # =========================
        assistant_message = {
            "role": "assistant",
            "content": response if response else None,
            "tool_calls": []
        }

        for index, call in tool_calls.items():

            assistant_message["tool_calls"].append({
                "id": call["id"],
                "type": "function",
                "function": {
                    "name": call["name"],
                    "arguments": call["arguments"]
                }
            })

        messages.append(assistant_message)

        print("ASSISTANT MESSAGE:")
        print(assistant_message)

        # =========================
        # RUN TOOL
        # =========================
        responses = restaurant_handle_tool_calls(
            assistant_message
        )

        print("TOOL RESPONSE:")
        print(responses)

        messages.extend(responses)

        # LOOP LAGI

In [31]:
def restaurant_handle_tool_calls(message):
    responses = []

    for tool_call in message["tool_calls"]:
        function_name = tool_call["function"]["name"]
        arguments = json.loads(
            tool_call["function"]["arguments"]
        )

        function = tools_map[function_name]
        result = function(**arguments)

        responses.append({
            "role": "tool",
            "content": str(result),
            "tool_call_id": tool_call["id"]
        })

    return responses

In [15]:
tools_map = {
    "get_all_menu": get_all_menu,
    "add_menu": add_menu,
    "get_price_by_name": get_price_by_name
}

In [39]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


CALLING OPENAI...
